<a href="https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before encoding my baseline rule, I check two February 2026 decision-time signals.

### Signal 1 — Search volume

I use February GSC impressions as a volume signal.

This is linked to the FlyRank quick-win logic: pages with meaningful observed search volume represent a more measurable opportunity than pages with very little observed visibility.

### Signal 2 — CTR relative to position

I compare CTR across February average-position buckets.

This is linked to the FlyRank CTR-fix logic because CTR should be interpreted together with search position rather than in isolation.

Each signal will receive one evidence-based verdict:

- `CONFIRMED`
- `OPPOSITE`
- `MIXED`
- `FALSE`

The verdicts are determined only after inspecting the observed bucket tables.

### Baseline rule

I will prioritize pages that have:

1. relatively high February search volume; and
2. relatively low February CTR.

The score is a simple, hand-designed combination:

`baseline_score = 0.60 × volume_score + 0.40 × low_ctr_score`

The weights are fixed by hand and are not fitted or learned from the data.

### Reason code

Every scored page receives:

`high_volume_low_ctr`

### Action

Every scored page receives:

`review_ctr`

The action is a recommendation for human review, not an automatic content change.

Only February 2026 decision-time information is used. No March data, future outcome, label, or label-derived feature is used.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import getpass
import duckdb

con = duckdb.connect()
HF_TOKEN = getpass.getpass("Enter your Hugging Face token: ")
# Authenticate with Hugging Face
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    );
""")


FEB = """
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-02/*.parquet'
    )
"""

feb = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_feb,

        SUM(gsc_clicks) AS clicks_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr_feb,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_sum_position) * 1.0
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_feb,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_days

    FROM ({FEB}) AS daily_performance

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Keep only pages with measurable February GSC data.
feb = feb[
    (feb["gsc_available_days"] > 0) &
    (feb["impressions_feb"] > 0) &
    (feb["ctr_feb"].notna()) &
    (feb["avg_position_feb"].notna())
].copy()

print(f"Page-level rows available: {len(feb):,}")

feb.head()

Enter your Hugging Face token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level rows available: 153,559


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,gsc_available_days
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,0.0,7.000000,2
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,0.2,4.200000,4
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,0.0,5.520833,26
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.0,6.391489,28
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,0.0,7.000000,13


### Signal 1 — Search volume
I bucket February GSC impressions into three groups.

For each bucket I show:

- `n` — number of pages;
- median impressions;
- median CTR.

The purpose is to check whether observed search volume provides a meaningful distinction between pages before using it in the baseline.

The verdict will be based on the actual observed table rather than being chosen in advance.

In [ ]:
signal1 = feb.copy()

signal1["volume_bucket"] = pd.qcut(
    signal1["impressions_feb"],
    q=3,
    labels=["LOW", "MEDIUM", "HIGH"],
    duplicates="drop"
)

volume_table = (
    signal1
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions_feb", "median"),
        median_ctr=("ctr_feb", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — SEARCH VOLUME")
print()
print(volume_table.to_string(index=False))

SIGNAL 1 — SEARCH VOLUME

volume_bucket     n  median_impressions  median_ctr
          LOW 51264                 5.0     0.00000
       MEDIUM 51127               119.0     0.00000
         HIGH 51168              1514.0     0.00196


### Signal 1 verdict: `[CONFIRMED]`

**Why:**

After running the bucket table, I observed that:

`Search volume is a useful prioritization signal. The HIGH-volume bucket has a much larger median number of impressions (1,514) than MEDIUM (119) and LOW (5), and it also has the highest median CTR (0.00196). This supports using search volume as part of the baseline rule because pages with greater search exposure have more potential value for CTR-focused review.`

Therefore, my verdict is:

**`[CONFIRMED]`**

### Signal 2 — CTR relative to position

CTR is not interpreted alone.

I bucket pages using their February average search position:

- `TOP_3`
- `4_TO_10`
- `11_TO_20`
- `21_PLUS`

For each bucket I calculate:

- `n` — number of pages;
- median CTR;
- median average position.

This checks the signal behind the FlyRank CTR-fix logic: whether CTR differs across search-position ranges.

The verdict will be based on the observed pattern.

In [ ]:
signal2.head()

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,gsc_available_days,position_bucket
0,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,0.0,7.000000,2,4_TO_10
1,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,0.2,4.200000,4,4_TO_10
2,client_3ffa76342f366962,content_cae1d5374958a649,96.0,0.0,0.0,5.520833,26,4_TO_10
3,client_3ffa76342f366962,content_dd66eecf9626cab8,235.0,0.0,0.0,6.391489,28,4_TO_10
4,client_3ffa76342f366962,content_c51f1e8ef5502177,18.0,0.0,0.0,7.000000,13,4_TO_10


In [ ]:
signal2 = feb.copy()

signal2["position_bucket"] = pd.cut(
    signal2["avg_position_feb"],
    bins=[0, 3, 10, 20, np.inf],
    labels=[
        "TOP_3",
        "4_TO_10",
        "11_TO_20",
        "21_PLUS"
    ],
    include_lowest=True
)

ctr_position_table = (
    signal2
    .groupby("position_bucket", observed=False)
    .agg(
        n=("content_hash_id", "size"),
        median_ctr=("ctr_feb", "median"),
        median_position=("avg_position_feb", "median")
    )
    .reset_index()
)

print("SIGNAL 2 — CTR RELATIVE TO POSITION")
print()
print(ctr_position_table.to_string(index=False))

SIGNAL 2 — CTR RELATIVE TO POSITION

position_bucket     n  median_ctr  median_position
          TOP_3 20174         0.0         2.000000
        4_TO_10 75602         0.0         6.210751
       11_TO_20 30811         0.0        13.375300
        21_PLUS 26972         0.0        32.428764


### Signal 2 verdict: `[MIXED]`

**Why:**

After comparing CTR across the position buckets, I observed:

`CTR relative to search position is a relevant signal for the baseline rule because CTR is evaluated within distinct position buckets (TOP_3, 4_TO_10, 11_TO_20, and 21_PLUS). However, the median CTR values round to 0.0 across all buckets, so this table does not show a strong separation in median CTR by position. I therefore treat the signal as MIXED as a useful diagnostic for CTR review, but not as evidence of a strong position–CTR relationship in this slice.`

Therefore, my verdict is:

**`[MIXED]`**

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The two signal checks above were completed before encoding the baseline.

I now encode one transparent, hand-designed rule.

The rule combines:

- 60% February search-volume rank;
- 40% inverse February CTR rank.

The weights are fixed manually and are not fitted to an outcome or label.

The rule produces:

- one score: `baseline_score`;
- one reason code: `high_volume_low_ctr`;
- one action label: `review_ctr`.

The queue is ranked from highest to lowest score.

No future data, March data, outcome label, or label-derived information is used.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

queue = feb.copy()

# Higher percentile rank = more observed search volume.
queue["volume_score"] = (
    queue["impressions_feb"]
    .rank(pct=True)
)

# Higher score = relatively lower CTR.
queue["low_ctr_score"] = (
    1 - queue["ctr_feb"].rank(pct=True)
)

# Transparent hand-designed score.
queue["baseline_score"] = (
    0.60 * queue["volume_score"]
    + 0.40 * queue["low_ctr_score"]
)

# One reason code and one action label.
queue["reason_code"] = "high_volume_low_ctr"
queue["action"] = "review_ctr"

# Rank highest score first.
queue = (
    queue
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)

queue["baseline_rank"] = np.arange(
    1,
    len(queue) + 1
)

print(f"Rows ranked: {len(queue):,}")

queue.head(10)

Rows ranked: 153,559


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,gsc_available_days,volume_score,low_ctr_score,baseline_score,reason_code,action,baseline_rank
0,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,0.0,0.070336,28,0.999987,0.679374,0.871742,high_volume_low_ctr,review_ctr,1
1,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,0.0,2.308282,28,0.999909,0.679374,0.871695,high_volume_low_ctr,review_ctr,2
2,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,0.0,19.067652,28,0.997831,0.679374,0.870448,high_volume_low_ctr,review_ctr,3
3,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,0.0,35.547883,28,0.997545,0.679374,0.870277,high_volume_low_ctr,review_ctr,4
4,client_23a62021009f63c4,content_1162dc8495e06dfb,19938.0,0.0,0.0,38.160949,28,0.993084,0.679374,0.867600,high_volume_low_ctr,review_ctr,5
5,client_23a62021009f63c4,content_c367b0ca57f3559b,19627.0,0.0,0.0,44.931829,28,0.992908,0.679374,0.867495,high_volume_low_ctr,review_ctr,6
6,client_23a62021009f63c4,content_67a19b4e8f52924e,19529.0,0.0,0.0,23.109427,28,0.992850,0.679374,0.867459,high_volume_low_ctr,review_ctr,7
7,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,18472.0,0.0,0.0,0.626353,28,0.992036,0.679374,0.866971,high_volume_low_ctr,review_ctr,8
8,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,15238.0,0.0,0.0,0.121932,28,0.988734,0.679374,0.864990,high_volume_low_ctr,review_ctr,9
9,client_23a62021009f63c4,content_73aa61dcedebbf30,15050.0,0.0,0.0,45.646777,28,0.988509,0.679374,0.864855,high_volume_low_ctr,review_ctr,10


### Write the ranked queue

The final ranked queue is written from the notebook to:

`work/outputs/baseline_action_score.csv`

In [ ]:
baseline_action_score = queue[
    [
        "baseline_rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "reason_code",
        "action",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "gsc_available_days"
    ]
].copy()

output_dir = Path("work/outputs")
output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    output_dir /
    "baseline_action_score.csv"
)

baseline_action_score.to_csv(
    output_path,
    index=False
)

print(f"Ranked rows: {len(baseline_action_score):,}")
print(f"CSV written to: {output_path}")

baseline_action_score.head(10)

Ranked rows: 153,559
CSV written to: work/outputs/baseline_action_score.csv


,baseline_rank,client_hash_id,content_hash_id,baseline_score,reason_code,action,impressions_feb,clicks_feb,ctr_feb,avg_position_feb,gsc_available_days
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.871742,high_volume_low_ctr,review_ctr,193954.0,0.0,0.0,0.070336,28
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,0.871695,high_volume_low_ctr,review_ctr,125035.0,0.0,0.0,2.308282,28
2,3,client_23a62021009f63c4,content_2f09787bdf392b16,0.870448,high_volume_low_ctr,review_ctr,34293.0,0.0,0.0,19.067652,28
3,4,client_23a62021009f63c4,content_559cdd76da9306de,0.870277,high_volume_low_ctr,review_ctr,32799.0,0.0,0.0,35.547883,28
4,5,client_23a62021009f63c4,content_1162dc8495e06dfb,0.867600,high_volume_low_ctr,review_ctr,19938.0,0.0,0.0,38.160949,28
5,6,client_23a62021009f63c4,content_c367b0ca57f3559b,0.867495,high_volume_low_ctr,review_ctr,19627.0,0.0,0.0,44.931829,28
6,7,client_23a62021009f63c4,content_67a19b4e8f52924e,0.867459,high_volume_low_ctr,review_ctr,19529.0,0.0,0.0,23.109427,28
7,8,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,0.866971,high_volume_low_ctr,review_ctr,18472.0,0.0,0.0,0.626353,28
8,9,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,0.864990,high_volume_low_ctr,review_ctr,15238.0,0.0,0.0,0.121932,28
9,10,client_23a62021009f63c4,content_73aa61dcedebbf30,0.864855,high_volume_low_ctr,review_ctr,15050.0,0.0,0.0,45.646777,28


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


I review the 20 highest-ranked pages.

For every page I record:

1. the action;
2. the reason code;
3. why the page is in the top 20;
4. what would make the recommendation wrong.

The purpose is to identify weak picks and expose limitations in the rule.

The rule is decision support rather than an automatic decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = baseline_action_score.head(20).copy()


def why_its_there(row):
    return (
        f"High baseline score ({row['baseline_score']:.3f}) "
        f"from February search volume "
        f"({row['impressions_feb']:,.0f} impressions) "
        f"combined with relatively low CTR "
        f"({row['ctr_feb']:.2%})."
    )


def confidence_note(row):
    return (
        "Moderate confidence: the recommendation is based on "
        "two observed February signals, but the rule does not "
        "account for query intent, SERP features, or snippet context."
    )


def what_would_make_it_wrong(row):
    return (
        "The low CTR may be explained by query intent, SERP features, "
        "brand/non-brand query mix, snippet wording, or normal search "
        "variation rather than a content problem."
    )


top20["why_its_there"] = top20.apply(
    why_its_there,
    axis=1
)

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

top20_review = top20[
    [
        "baseline_rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "why_its_there",
        "what_would_make_it_wrong"
    ]
].copy()

top20_review

,baseline_rank,content_hash_id,action,reason_code,confidence_note,why_its_there,what_would_make_it_wrong
0,1,content_fec55986a1868d62,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.872) from February sear...,"The low CTR may be explained by query intent, ..."
1,2,content_c9f840183215651b,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.872) from February sear...,"The low CTR may be explained by query intent, ..."
2,3,content_2f09787bdf392b16,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.870) from February sear...,"The low CTR may be explained by query intent, ..."
3,4,content_559cdd76da9306de,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.870) from February sear...,"The low CTR may be explained by query intent, ..."
4,5,content_1162dc8495e06dfb,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.868) from February sear...,"The low CTR may be explained by query intent, ..."
5,6,content_c367b0ca57f3559b,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.867) from February sear...,"The low CTR may be explained by query intent, ..."
6,7,content_67a19b4e8f52924e,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.867) from February sear...,"The low CTR may be explained by query intent, ..."
7,8,content_1cb7263083e97ba1,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.867) from February sear...,"The low CTR may be explained by query intent, ..."
8,9,content_d16bbebfbb3c8fda,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.865) from February sear...,"The low CTR may be explained by query intent, ..."
9,10,content_73aa61dcedebbf30,review_ctr,high_volume_low_ctr,Moderate confidence: the recommendation is bas...,High baseline score (0.865) from February sear...,"The low CTR may be explained by query intent, ..."


In [ ]:
for _, row in top20_review.iterrows():

    print(
        f"{row['baseline_rank']}. "
        f"{row['content_hash_id']}"
    )

    print(
        f"   Action: {row['action']}"
    )

    print(
        f"   Reason code: {row['reason_code']}"
    )

    print(
        f"   Confidence: {row['confidence_note']}"
    )

    print(
        f"   Why it's there: {row['why_its_there']}"
    )

    print(
        f"   What would make it wrong: "
        f"{row['what_would_make_it_wrong']}"
    )

    print()

1. content_fec55986a1868d62
   Action: review_ctr
   Reason code: high_volume_low_ctr
   Confidence: Moderate confidence: the recommendation is based on two observed February signals, but the rule does not account for query intent, SERP features, or snippet context.
   Why it's there: High baseline score (0.872) from February search volume (193,954 impressions) combined with relatively low CTR (0.00%).
   What would make it wrong: The low CTR may be explained by query intent, SERP features, brand/non-brand query mix, snippet wording, or normal search variation rather than a content problem.

2. content_c9f840183215651b
   Action: review_ctr
   Reason code: high_volume_low_ctr
   Confidence: Moderate confidence: the recommendation is based on two observed February signals, but the rule does not account for query intent, SERP features, or snippet context.
   Why it's there: High baseline score (0.872) from February search volume (125,035 impressions) combined with relatively low CTR (0.0

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The ranked list should not be treated as ground truth.

A weak pick can occur because low CTR has multiple possible explanations that are not represented in this simple baseline.

For example:

- query intent;
- SERP features;
- brand versus non-brand searches;
- snippet wording;
- normal search variation.

Therefore, a high baseline score means "review this page first", not "this page definitely has a problem."

At least one top-ranked page should be identified as a potentially weak pick during manual review.

### Leakage check

The baseline score uses only February 2026 decision-time information:

- `impressions_feb`
- `ctr_feb`

It does not use:

- March or later data;
- future outcomes;
- `went_dark`;
- labels;
- label-derived features.

Therefore, the ranked queue is generated without future-window information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Explicit audit of the inputs used by the baseline score.

score_inputs = {
    "impressions_feb",
    "ctr_feb"
}

forbidden_inputs = {
    "went_dark",
    "label",
    "target",
    "gsc_clicks_mar",
    "gsc_impressions_mar",
    "ctr_mar",
    "position_mar",
    "march"
}

# These are the actual columns referenced by the score.
actual_score_columns = {
    "impressions_feb",
    "ctr_feb"
}

leakage_found = (
    actual_score_columns
    .intersection(forbidden_inputs)
)

print("Actual score inputs:")
for col in sorted(actual_score_columns):
    print(f"  - {col}")

print("\nForbidden future/label inputs:")
for col in sorted(forbidden_inputs):
    print(f"  - {col}")

print(
    "\nForbidden inputs used by score:",
    leakage_found
)

assert len(leakage_found) == 0

print("\nLeakage check: PASSED")
print(
    "The baseline score uses February decision-time "
    "information only."
)

Actual score inputs:
  - ctr_feb
  - impressions_feb

Forbidden future/label inputs:
  - ctr_mar
  - gsc_clicks_mar
  - gsc_impressions_mar
  - label
  - march
  - position_mar
  - target
  - went_dark

Forbidden inputs used by score: set()

Leakage check: PASSED
The baseline score uses February decision-time information only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.